# U(1)-Gauged Dirac Spectra in 3D / 4D: Wilson vs Brillouin

Eigenvalue spectra of the **Wilson** `(lap_std, der_std)` and **Brillouin** `(lap_bri, der_iso)`
lattice Dirac operators on **genuinely thermalized** U(1) gauge backgrounds in 3D and 4D.

The `g-koutsou/u1-hmc` generator only produces 2D configurations, so here we generate the
gauge field ourselves with a **dimension-general red-black von Mises heatbath**
(`gauge.synthetic.heatbath_links` / `generate_heatbath_ensemble`). These are *bona fide*
equilibrium configurations at coupling `β` — not random perturbations:

- in 2D the mean plaquette matches the exact `I₁(β)/I₀(β)`;
- in 3D/4D it sits above that (the plaquettes are coupled — correct physics).

The configurations are cached under `data_files/raw/` (via `save_config`) so they are reusable.

**What to look for** (cf. S. Dürr & G. Koutsou, *Phys. Rev. D* **83**, 114512, 2011):
- **Wilson** spans `Re λ ∈ [0, 2d]` with `d+1` branches (3 in 3D, 5 in 4D) of alternating chirality.
- **Brillouin** always stays in `Re λ ∈ [0, 2]`, condensing the unphysical modes near `Re λ ≈ 2`
  and leaving the physical branch near `Re λ ≈ 0` well separated.
- The free-field branches (blue `+`) are the skeleton; the gauge background fills the interior
  while the extreme tips retract inward (additive mass renormalization).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lattice_dirac_spectra.constants.operators import operator_keys
from lattice_dirac_spectra.constants.paths import RAW_DATA_DIR, PLOTS_DIR
from lattice_dirac_spectra.gauge.synthetic import generate_heatbath_ensemble
from lattice_dirac_spectra.gauge.u1_config import list_trajectories, load_ensemble
from lattice_dirac_spectra.gauge.observables import plaquette
from lattice_dirac_spectra.spectra.gauged import gauged_spectrum
from lattice_dirac_spectra.spectra.free_field import eigenvalues_from_matrix
from lattice_dirac_spectra.visualization.style import apply_style

%matplotlib inline
apply_style()
plt.rcParams.update({"figure.dpi": 110})

## 1. Settings

`DIM = 3` is light; `DIM = 4` is heavier — the eigensolve is on a `(L^d · N_s)`-square matrix.
Recommended starting points: **3D** `L = 6`, **4D** `L = 4`. Bump `L`/`N_CONFIGS` once you've
confirmed it runs. Larger `β` is colder (cloud hugs the free-field branches); smaller `β` is
rougher (cloud fills out more).

In [ ]:
# ── edit this cell ────────────────────────────────────────────────────────────
DIM = 4  # 3 or 4
L = 4  # linear extent  (3D: try 6 ; 4D: try 4, then 6 if patient)
BETA = 4.0  # gauge coupling (larger = smoother field)

N_CONFIGS = 6  # thermalized configurations to stack
N_THERM = 150  # heatbath sweeps for initial thermalization
N_DECORR = 15  # sweeps between stored configurations
SEED = 0
N_APE = 0  # APE smearing steps applied at diagonalization time

REGENERATE = False  # True forces regeneration even if the cached file exists
# ─────────────────────────────────────────────────────────────────────────────

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_FILE = RAW_DATA_DIR / f"u1_d{DIM}_L{L}_beta{BETA:5.3f}.h5"
print(f"Ensemble file : {ENSEMBLE_FILE}")
print(f"Lattice       : {DIM}D, L = {L}, beta = {BETA}")

## 2. Generate (or load) the thermalized ensemble

The heatbath runs once and the result is cached. Re-running this cell reloads the cached file
unless `REGENERATE = True`.

In [ ]:
need_gen = REGENERATE or not ENSEMBLE_FILE.exists()
if need_gen:
    print("Generating ensemble via heatbath … (this is the slow step the first time)")
    generate_heatbath_ensemble(
        str(ENSEMBLE_FILE),
        d=DIM,
        L=L,
        beta=BETA,
        n_configs=N_CONFIGS,
        n_therm=N_THERM,
        n_decorr=N_DECORR,
        rng=np.random.default_rng(SEED),
    )
    print("done.")
else:
    print("Using cached ensemble.")

configs = list(load_ensemble(str(ENSEMBLE_FILE), BETA))
plaqs = [c.metadata["plaquette"] for c in configs]
print(f"Loaded {len(configs)} configs  |  d = {configs[0].dim}  |  L = {configs[0].L}")
print(f"Mean plaquette = {np.mean(plaqs):.4f} ± {np.std(plaqs):.4f}")

## 3. Spectral cloud — Wilson vs Brillouin

Gauged eigenvalues stacked over the ensemble (red), with the free-field branches overlaid (blue `+`).

In [ ]:
KERNELS = ["Wilson", "Brillouin"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, kernel in zip(axes, KERNELS):
    der, lap = operator_keys(kernel)

    all_eigs = np.concatenate(
        [gauged_spectrum(lap, der, c, n_ape=N_APE) for c in configs]
    )
    free = eigenvalues_from_matrix(lap, der, d=DIM, L=L)

    ax.scatter(
        all_eigs.real,
        all_eigs.imag,
        s=2,
        c="#d94f4f",
        alpha=0.30,
        linewidths=0,
        label=f"gauged ({len(configs)} cfg, {len(all_eigs)} eigs)",
    )
    ax.plot(
        free.real,
        free.imag,
        "+",
        ms=6,
        c="#3a7fc1",
        mew=1.2,
        alpha=0.65,
        label="free field (links = 1)",
    )

    ax.axhline(0, color="k", lw=0.5, alpha=0.20)
    ax.axvline(0, color="k", lw=0.5, alpha=0.20)
    ax.set_xlabel(r"$\mathrm{Re}\,\lambda$")
    ax.set_ylabel(r"$\mathrm{Im}\,\lambda$")
    ax.set_title(f"{kernel} kernel", fontweight="bold")
    ax.set_aspect("equal")
    ax.legend(fontsize=8, loc="upper right")
    ax.text(
        0.02,
        0.97,
        f"mean plaq = {np.mean(plaqs):.4f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", alpha=0.8, ec="#cccccc"),
    )

fig.suptitle(
    rf"$\beta = {BETA}$  |  {DIM}D U(1) heatbath  |  $L = {L}$  |  "
    rf"$N_{{\rm cfg}} = {len(configs)}$  |  $N_{{\rm APE}} = {N_APE}$",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))

# Optional: save a PDF under output/plots
# PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# fig.savefig(PLOTS_DIR / f"gauged_cloud_{DIM}d_L{L}_beta{BETA:5.3f}.pdf", bbox_inches="tight")

plt.show()

## 4. Notes and next steps

**Cost.** The heatbath sweeps are cheap; the bottleneck is the dense eigensolve on a
`(L^d · N_s)`-square matrix (`N_s = 2` in 2D/3D, `4` in 4D). `4D L=4 → 1024²` (fast),
`4D L=6 → 5184²` (minutes), `4D L=8` is infeasible dense — use the sparse `k`/`sigma` path in
`gauged_spectrum` to extract a slice near a shift instead.

**Knobs.**
- **`BETA`** — colder (larger) `β` tightens the cloud onto the free-field branches; rougher
  (smaller) `β` fills the interior. Try `β = 1, 2, 4`.
- **`N_APE`** — a few APE smearing steps smooth the background and tighten the branches.
- **`REGENERATE`** — the ensemble is cached under `data_files/raw/`; flip this to rebuild.

**Sanity checks.**
- The Wilson real-axis extent is exactly `2d` (6 in 3D, 8 in 4D); Brillouin is always `[0, 2]`.
- The spectrum is closed under complex conjugation (γ₅-Hermiticity) and is gauge invariant.
- Setting `BETA` very large should drive the cloud onto the free-field `+` markers.

**Caveat on the background.** The heatbath gives a genuine pure-gauge U(1) equilibrium ensemble.
It is *not* the same as a 4D ensemble from a full QCD/dynamical-fermion simulation, but for
studying the **operator's** spectral structure (the goal here) the pure-gauge background is the
right and sufficient probe.